# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [7]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0,0)
        self.walls = {
            (0,3),
            (1,1),
            (2,4),
            (4,2)
        }
        self.slippery_states = {
            (1,2),
            (2,1),
            (3,3)

        }

        self.terminal_states = {
            (2,2): 2,
            (0,5): 10,
            (3,5): -10
            # (row, col): reward
        }

        self.danger_states = {
            (4,1): -3,
            (1,4): -3
            # (row, col): -3
        }

        self.living_reward = -1.0 # Original
        # self.living_reward = -0.1 # Experimento A
        # self.gamma = 0.9 # Original
        self.gamma = 0.99 # Experimento C

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _move(self, state, action):
        nxt = (state[0] + action[0], state[1] + action[1])
        return nxt if self.is_valid_state(nxt) else state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_main, p_side = 0.60, 0.20  # Original
            # p_main, p_side = 0.40, 0.30  # Experimento B

        else:
            p_main, p_side = 0.90, 0.05

        dr, dc = action
        left = (-dc, dr)    # giro de 90° a la izquierda de la acción
        right = (dc, -dr)   # giro de 90° a la derecha de la acción

        probs = {}
        for a, p in [(action, p_main), (left, p_side), (right, p_side)]:
            nxt = self._move(state, a)
            probs[nxt] = probs.get(nxt, 0.0) + p
        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [8]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [9]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(p * V[s_next] for s_next, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for k in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                # Estado absorbente: su valor es solo su recompensa
                V_new[s] = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
                V_new[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, k

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    return {
        s: max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
        for s in grid.states()
        if not grid.is_terminal(s)
    }



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [10]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for _ in range(max_iter):
        V_new = {}
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                # Estado absorbente: su valor es solo su recompensa
                V_new[s] = grid.get_reward(s)
            else:
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum T(s,a,s') V^pi(s')  (mismo argmax que en Value Iteration)
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: siempre UP
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []  # número de estados que cambiaron de acción en cada iteración

    for _ in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        changes = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changes)
        policy = new_policy
        # 4. repetir hasta estabilidad
        if changes == 0:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [11]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [12]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 24

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [17, 7, 2, 2, 0]

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

> Respuestas
> 1. El robot prefiere la estacion de carga
> 2. Existen menores riesgos de alcanzar esa recompensa, y además el camino es menor resultando en un menor castigo de movimiento (-1 por paso)
> 3. En los estados alrededor de el piso resbaloso.
> 4. Incentivar caminos más directos
> 5. Las distintas probabilidades se ven actualizadas dependiendo de las recompensas de sus decisiones


### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

> Predicción: El robot puede terminar prefiriendo llegar al +10
> Realidad? Correcto

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

> Predicción: El robot preferirá evitar más los pisos resbalosos; el valor de esos estados será menor
> Realidad: Efectivamentente el valor disminuyó en esos estados


### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

> Predicción: Si
> Realidad: si

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [13]:
# BONUS — Bisección sobre living_reward
# Se usa el modelo original (gamma = 0.9), independientemente del experimento activo.

def destino_desde_start(living_reward, gamma=0.9, max_steps=100):
    """Resuelve el MDP y sigue la política (movimiento deseado, sin desviaciones)
    desde START. Devuelve la recompensa del terminal al que llega, o None si entra en un ciclo."""
    g = WarehouseMDP()
    g.living_reward = living_reward
    g.gamma = gamma

    V, _ = value_iteration(g)
    pi = extract_policy(g, V)

    s = g.start
    for _ in range(max_steps):
        if g.is_terminal(s):
            return g.get_reward(s)
        s = g._move(s, pi[s])
    return None


def biseccion_living_reward(lo=-1.0, hi=-0.1, tol=1e-3):
    # lo debe ir a la carga (+2) y hi a la entrega (+10)
    assert destino_desde_start(lo) == 2, "lo no va a la carga +2"
    assert destino_desde_start(hi) == 10, "hi no va a la entrega +10"

    while hi - lo > tol:
        mid = (lo + hi) / 2
        if destino_desde_start(mid) == 10:
            hi = mid
        else:
            lo = mid
    return lo, hi


lo, hi = biseccion_living_reward()
print(f"Con living_reward <= {lo:.4f} el robot va a la carga +2")
print(f"Con living_reward >= {hi:.4f} el robot va a la entrega +10")
print(f"Umbral aproximado: {(lo + hi) / 2:.4f}")


Con living_reward <= -0.7987 el robot va a la carga +2
Con living_reward >= -0.7979 el robot va a la entrega +10
Umbral aproximado: -0.7983
